GENERAL FUNCTIONS FOR GF

In [ ]:
import numpy as np



def poly_mult_gf2(p1, p2):
    """Multiplies two polynomials in GF(2) using Shift and XOR."""
    # very heavy but with no constraint, we will use it only to calculate g at the beginning
    result = 0
    while p2 > 0:
        if p2 & 1:  
            result ^= p1  # XOR
        p1 <<= 1      # (shift left)
        p2 >>= 1      # (shift right)
        
    return result

def poly_div_gf2(dividend, divisor):
    """Performs polynomial division in GF(2) and returns the remainder."""
    divident_i = dividend
    divisor_len = divisor.bit_length()

    while dividend.bit_length() >= divisor_len:
        shift_amount = dividend.bit_length() - divisor_len
        aligned_divisor = divisor << shift_amount
        dividend ^= aligned_divisor

    if dividend.bit_length() >= divisor_len:
        raise ValueError("Unexpected error: Dividend should be smaller than divisor at this point.")
    else:
        result =  divident_i ^ dividend
    return result



# if __name__ == "__main__":
#     msg = 0b101101000   # Messaggio: 45
#     divisor = 0b1101   # Divisore (Generatore): 13
#     parity = poly_div_gf2(msg, divisor)
#     print(f"Dividend: {bin(msg)}")
#     print(f"Divisor:              {bin(divisor)}")
#     print(f"Rest:        {bin(parity)[2:].zfill(divisor.bit_length() - 1)}")

# (this tables are explained in the Decoding sec)
exp_table = [1, 2, 4, 3, 6, 7, 5,   1, 2, 4, 3, 6, 7, 5] 
log_table = [0, 0, 1, 3, 2, 6, 4, 5]

def gf_mult(a, b):
    if a == 0 or b == 0:
        return 0
    # Aggiunto il modulo 8191
    index = (log_table[a] + log_table[b]) % 8191 # (o usiamo il modulo a facciamo la LUT doppia)
    return exp_table[index]

def gf_inv(a):
    if a == 0:
        raise ZeroDivisionError("0 non ha inverso in GF")
    return exp_table[(8191 - log_table[a]) % 8191]





ENCODING FUNCTIONS

In [ ]:
def g_polynomial(t=2):

    m_pol_1 = 0b1011  # x^3 + x + 1
    m_pol_3 = 0b1101  # x^3 + x^2 + 1
    
    if t == 1:
        return m_pol_1
    elif t == 2:
        return poly_mult_gf2(m_pol_1, m_pol_3)
    else:
        raise ValueError("t must be < 3")
    
# Test the code
# g = g_polynomial(2)
# print(f"The generator for t=2 is: {bin(g)}")

def encoding(word, g, t):
    
    word_s = word << (3 * t)
    word_f = poly_div_gf2(word_s, g)
    return word_f

DECODING FUNCTIONS -- Berlekamp-Massey

In [ ]:
# starting with the syndromes calculations, using the Horner method
# the lenght of the LUT strictly depends on m
# for m = 13 P(x) = x^13 + x^4 + x^3 + x + 1 (1010000000001 in binary)

def generate_gf13_header(filename="gf13_luts.h"):
    m = 13
    size = (1 << m) - 1  # 8191
    
    # Primitive Polynomial: x^13 + x^4 + x^3 + x + 1
    # 0x201B represents the polynomial including the x^13 bit
    poly = 0x201B 
    
    exp_table = [0] * (size * 2)
    log_table = [0] * (size + 1)
    
    current = 1
    for i in range(size):
        exp_table[i] = current
        log_table[current] = i
        
        # Step to next power: alpha^(i+1)
        current <<= 1
        if current & (1 << m): # If overflow bit 13 is set
            current ^= poly
            
    # Duplicate the exp_table for the "no-modulo" multiplication trick
    for i in range(size):
        exp_table[i + size] = exp_table[i]
        
    # Write to C++ Header File
    with open(filename, "w") as f:
        f.write("#ifndef GF13_LUTS_H\n#define GF13_LUTS_H\n\n")
        f.write("#include <stdint.h>\n\n")
        
        # Write Exp Table
        f.write(f"const uint16_t exp_table[{len(exp_table)}] = {{\n    ")
        for i, val in enumerate(exp_table):
            f.write(f"{val},")
            if (i + 1) % 12 == 0: f.write("\n    ")
        f.write("\n};\n\n")
        
        # Write Log Table
        f.write(f"const uint16_t log_table[{len(log_table)}] = {{\n    ")
        for i, val in enumerate(log_table):
            f.write(f"{val},")
            if (i + 1) % 12 == 0: f.write("\n    ")
        f.write("\n};\n\n")
        
        f.write("#endif // GF13_LUTS_H\n")
    
    print(f"Header file '{filename}' generated successfully.")

if __name__ == "__main__":
    generate_gf13_header()


# EXAMPLE WITH M = 3
# Ex: alpha^2 = 4. (because  001 -> a^0, 010 -> a^1, 100 -> a^2,  011 -> a^3, 110 -> a^4, 101 -> a^5, 111 -> a^6)
exp_table = [1, 2, 4, 3, 6, 7, 5,   1, 2, 4, 3, 6, 7, 5] 

# Ex: alpha^3. log_table[4] = 2.
# Index 0 is not defined (log of 0 is undefined), so we can set it to 0 or -1 as a placeholder.
log_table = [0, 0, 1, 3, 2, 6, 4, 5]

def calculate_horner(msg, root_idx):
    root = exp_table[root_idx]
    s_i = 0
    msg_bits = msg.bit_length()
    for bit_pos in range(msg_bits - 1, -1, -1): # most to least significant bit
            # Estraiamo il singolo bit
            bit = (msg >> bit_pos) & 1
            # METODO DI HORNER: Sindrome = (Sindrome_precedente * radice) XOR bit
            s_i = gf_mult(s_i, root) ^ bit
    return s_i

def calculate_syndromes(msg, t):
    """Calculates the syndromes S_1, S_2, ..., S_{2t} for a received message. """
    s1 = calculate_horner(msg, root_idx=1)
    s2 = gf_mult(s1, s1)
    if t == 1:
        return [s1, s2]
    else:
        s3 = calculate_horner(msg, root_idx=3)
        s4 = gf_mult(s2, s2)
        return [s1, s2, s3, s4]


if __name__ == "__main__":
    
    perfect_codeword = 0b1010011
    corrupted_codeword = 0b1010010
    print("No errors")
    s_perf = calculate_syndromes(perfect_codeword, t=2)
    print(f"Sindromi: S_1 = {s_perf[0]}, S_2 = {s_perf[1]}, S_3 = {s_perf[2]}, S_4 = {s_perf[3]}")
    print("\nCorrupted")
    s_corr = calculate_syndromes(corrupted_codeword, t=2)
    print(f"Sindromi: S_1 = {s_corr[0]}, S_2 = {s_corr[1]}, S_3 = {s_corr[2]}, S_4 = {s_corr[3]}")


## LOCALIZATOR POLYNOMIAL LAMBDA (using Berlekamp-Massey Algorithm)

def bch_berlekamp_massey(syndromes, t=2):
    """
    syndromes: [S1, S2, S3, S4]
    Returns lambda coefficients
    """
    lambda_poly = [1]        # starting point
    b_poly = [1]             # support polynomial
    L = 0                    # current degree of lambda

    for k in range(1, (2*t) + 1):
        # d: discrepancy
        # d = S_k + sum(Lambda_i * S_{k-i})
        d = syndromes[k-1]
        for i in range(1, L + 1):
            if i < len(lambda_poly): # se è più corto significa che non considera degli zeri in coda
                term = gf_mult(lambda_poly[i], syndromes[k-1-i]) # remember that k-1-i + i = k-1
                d ^= term
        # Shift B(x) = B(x) * x
        b_poly.insert(0, 0) # B(x) must grow with time
        
        if d != 0:
            T = list(lambda_poly)
            
            # check for length and pad with zeros if necessary
            max_len = max(len(lambda_poly), len(b_poly)) 
            lambda_poly += [0] * (max_len - len(lambda_poly)) # pad with zeros if necessary
            
            # 2. update: Lambda(x) = Lambda(x) + d * B(x)
            for i in range(len(b_poly)):
                lambda_poly[i] ^= gf_mult(d, b_poly[i])
            
            if 2 * L < k:
                # 3. update B(x) = Lambda(x) / d
                L = k - L
                d_inv = gf_inv(d)
                b_poly = [gf_mult(val, d_inv) for val in T]
                
    # Pulisce gli zeri in coda e limita a t+1
    lambda_poly += [0] * ((t + 1) - len(lambda_poly))
    return lambda_poly[:t+1]



Header file 'gf13_luts.h' generated successfully.
No errors
Sindromi: S_1 = 0, S_2 = 0, S_3 = 3, S_4 = 0

Corrupted
Sindromi: S_1 = 1, S_2 = 1, S_3 = 2, S_4 = 1
